In [ ]:
from IPython.utils import io
import tqdm.notebook
import os, sys, random
total = 100
with tqdm.notebook.tqdm(total=total) as pbar:
    with io.capture_output() as captured:
      # Instalar rdkit
      !pip -q install rdkit
      pbar.update(50)
      # Instalar molvs
      !pip install -q molvs
      pbar.update(100)

  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
import rdkit
print(f"rdkit_version: {rdkit.__version__}")

rdkit_version: 2025.09.6


In [ ]:
#The code requires a column with the smiles strings and the header named as: Smiles

#pip install molvs
import pandas as pd
import rdkit
from rdkit import Chem
from rdkit.Chem import PandasTools

from rdkit.Chem import SanitizeMol
from rdkit.Chem import RemoveHs
from rdkit.Chem import AssignStereochemistry

from rdkit.Chem import inchi

#from molvs.standardize import Standardizer
from molvs.metal import MetalDisconnector
from molvs.fragment import LargestFragmentChooser
from molvs.normalize import Normalizer
from molvs.charge import Reionizer, Uncharger

from molvs import validate_smiles
#from molvs import standardize_smiles

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Compuestos
url_data = "https://drive.google.com/file/d/10H2aHUABrWz93QAk3MeYXhYw7w4-5NJ1/view?usp=sharing"
url_data ='https://drive.google.com/uc?id=' + url_data.split('/')[-2]
df = pd.read_csv(url_data)
df

,SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical
0,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NCC1=CC=CC3=C1C...,3/000000487,InChI=1S/C21H17N3O5/c25-16(22-10-12-4-2-5-14-1...,C21H17N3O5,391,1.84,5.39,-8.34,6.45,-60
1,ClC1=CC=C(C=C1)COCCNC(=O)C/C=C\1/NC2C(C1)=CC=C...,3/000000194,InChI=1S/C22H25ClN2O2/c1-2-16-3-6-18-14-20(25-...,C22H25N2O2Cl,384,4.17,4.95,-7.42,6.45,-100
2,C1C(OCC2=NC=CC(=C2)C(=O)N)=CC(=CC=1)N1C(=CCNC1...,3/000000246,InChI=1S/C18H18N4O4/c19-17(23)12-4-7-21-13(8-1...,C18H17N4O4,353,0.87,4.92,-2.59,22.40,-180
3,C1=NC2C(C(=O)O1)=CC=CC=2NC(=O)C/C=C\1/NC2C(C1)...,3/000000066,InChI=1S/C21H19N3O3/c1-2-13-6-7-14-11-15(23-18...,C21H19N3O3,361,2.80,4.90,-6.57,6.45,-100
4,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NC[C@@H](CO)F)/C2,3/000000466,InChI=1S/C15H17FN2O4/c16-10(8-19)7-17-13(20)5-...,C15H17N2O4F,308,0.96,4.90,-2.45,4.94,-190
...,...,...,...,...,...,...,...,...,...,...
295,C(=C\CN)/C1=CC=C2C(=C1)C(=O)C=CN2,6/000000083,InChI=1S/C12H12N2O/c13-6-1-2-9-3-4-11-10(8-9)1...,C12H12N2O,200,1.51,3.01,-9.66,3.58,-80
296,OC(=O)C/C=C/C1C=CC=C(C=1Cl)O,4/000000101,InChI=1S/C10H9ClO3/c11-10-7(3-1-5-8(10)12)4-2-...,C10H9O3Cl,212,2.27,3.01,-2.70,2.22,0
297,OC(=O)/C=C/CCC1=NC2=C(N1)N=CC=C2,4/000000122,InChI=1S/C11H11N3O2/c15-10(16)6-2-1-5-9-13-8-4...,C11H11N3O2,217,1.00,3.01,-10.12,4.81,-100
298,OC1=CC=CC(=C1N1C=C[N]C1=[NH2])I,2/000000087,InChI=1S/C9H8IN3O/c10-6-2-1-3-7(14)8(6)13-5-4-...,C9H8N3OI,301,1.93,3.01,-6.51,13.40,-120


In [ ]:
#NOTE: If is intended to determine the canonical tautomer it is important to take into account that the rdkit and molvs functions for this purpose
#sometimes don't preserve the stereochemistry, particularly it is adviced to use molvs over rdkit, since molvs presents this problem with less frequency.

def curation(x): # Modified the arguments, as the initial call was with 1 argument.
    try:
        mol = Chem.MolFromSmiles(x)  # Correct indentation, use argument x to create the mol object
        if mol is None: # check if mol is None, to avoid the exception
            #If rdkit could not parse the smiles, returns Error 1
            return "Error 1"
        else:
            # This block will only run if the initial MolFromSmiles returns an object
            mol = Chem.MolFromSmiles(x, sanitize=True) # After constructing the molecule to a format where rdkit can work with the molecule, it is applied the function Chem.SanitizeMol() By default sanitize is always True, but here is shown this parameter to remember that this function includes the sanitize function
            #Chem.SanitizeMol(mol)                                   # Modifies directly the molecule that is why is not neccesary to include 'mol=' .The sanitize function makes the following: kekulize, check valencies, set aromaticity, conjugation and hybridization. Not recommended to not use sanitize, because in some cases the stereochemistry is not preserved without using sanitize.
            mol = Chem.RemoveHs(mol)                                 # Removal of explicit hydrogens. Some rdkit functions may trouble with molecules with explicit hydrogens such as some functions to calculate physicochemical properties
            mol = MetalDisconnector().disconnect(mol)                # Disconnects metal atoms that are covalently bonded to non-metals.
            mol = LargestFragmentChooser().choose(mol)               # From the molecules that used to be connected with metals or another salts, just the largest fragment is kept.

            allowed_elements = {"H","B","C","N","O","F","Si","P","S","Cl","Se","Br","I"}
            actual_elements = set([atom.GetSymbol() for atom in mol.GetAtoms()])
            if len(actual_elements-allowed_elements) == 0:

              mol = Normalizer().normalize(mol)                        # Apply a series of Normalization transforms to correct functional groups and recombine charges.
              mol = Reionizer().reionize(mol)                          # Ensure the strongest acid groups protonate first in partially ionized molecules.
              mol = Uncharger().uncharge(mol)                          # Attempts to neutralize the molecules. Partially ionized molecules will remain without changes.
              Chem.AssignStereochemistry(mol, force=True, cleanIt=True) # Modifies directly the molecule that is why is not neccesary to include 'mol=' .Recalculation of stereochemistry to assure that the original stereochemistry is preserved.
              mol = Chem.MolToSmiles(mol, isomericSmiles=True)                               # Returns the canonical SMILES string for a molecule. The canonical strings generated by rdkit may be different from the canonical SMILES generated by other software. The generated canonical SMILES must be comparable to the ones in ChEMBL since the SMILES strings in this database were generated with rdkit: https://chembl.gitbook.io/chembl-interface-documentation/frequently-asked-questions/drug-and-compound-questions#how-are-the-smiles-and-inchi-created-for-chembl
                                                                                       # By default isomericSmiles is True, nonetheless the parameter is preserved to remember that can be recovered the smiles without the stereochemistry information.
            #  print(f"{y} processed row")                                                    # Print the index of the processed row
              return mol
            else:
                # If molecule contains other than the allowed elements, return "Error 2"
                return "Error 2"

    except Exception as e:
        print(f"Error processing SMILES: {x}. Error message: {e}") # better exception handling
        return "Something else was found"

# Alternative ways to call the molvs functions.
#MetalDisconnector as example.
# mol = self.disconnect_metals(mol) #In the standardizer class was defined MetalDisconnector as disconnect_metals, the remaining classes also were redifined
# mol = MetalDisconnector()(mol)

In [ ]:
df["SMILES_curated"] = [curation(smi) for smi in df["SMILES"]]
df

[23:24:45] Explicit valence for atom # 14 N, 4, is greater than permitted
[23:24:45] Explicit valence for atom # 15 N, 4, is greater than permitted
[23:24:45] Explicit valence for atom # 5 N, 4, is greater than permitted
[23:24:45] Explicit valence for atom # 0 N, 4, is greater than permitted
[23:24:45] Explicit valence for atom # 11 N, 4, is greater than permitted
[23:24:45] Explicit valence for atom # 0 N, 4, is greater than permitted
[23:24:45] Explicit valence for atom # 11 N, 4, is greater than permitted
[23:24:45] Explicit valence for atom # 0 N, 4, is greater than permitted
[23:24:45] Explicit valence for atom # 12 N, 4, is greater than permitted


,SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical,SMILES_curated
0,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NCC1=CC=CC3=C1C...,3/000000487,InChI=1S/C21H17N3O5/c25-16(22-10-12-4-2-5-14-1...,C21H17N3O5,391,1.84,5.39,-8.34,6.45,-60,O=C(C/C=C1\Cc2cccc(C(=O)O)c2N1)NCc1cccc2c1C(=O...
1,ClC1=CC=C(C=C1)COCCNC(=O)C/C=C\1/NC2C(C1)=CC=C...,3/000000194,InChI=1S/C22H25ClN2O2/c1-2-16-3-6-18-14-20(25-...,C22H25N2O2Cl,384,4.17,4.95,-7.42,6.45,-100,CCc1ccc2c(c1)N/C(=C/CC(=O)NCCOCc1ccc(Cl)cc1)C2
2,C1C(OCC2=NC=CC(=C2)C(=O)N)=CC(=CC=1)N1C(=CCNC1...,3/000000246,InChI=1S/C18H18N4O4/c19-17(23)12-4-7-21-13(8-1...,C18H17N4O4,353,0.87,4.92,-2.59,22.40,-180,NC(=O)c1ccnc(COc2cccc(N3CNCC=C3C(=O)O)c2)c1
3,C1=NC2C(C(=O)O1)=CC=CC=2NC(=O)C/C=C\1/NC2C(C1)...,3/000000066,InChI=1S/C21H19N3O3/c1-2-13-6-7-14-11-15(23-18...,C21H19N3O3,361,2.80,4.90,-6.57,6.45,-100,CCc1ccc2c(c1)N/C(=C/CC(=O)Nc1cccc3c(=O)ocnc13)C2
4,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NC[C@@H](CO)F)/C2,3/000000466,InChI=1S/C15H17FN2O4/c16-10(8-19)7-17-13(20)5-...,C15H17N2O4F,308,0.96,4.90,-2.45,4.94,-190,O=C(C/C=C1\Cc2cccc(C(=O)O)c2N1)NC[C@H](F)CO
...,...,...,...,...,...,...,...,...,...,...,...
295,C(=C\CN)/C1=CC=C2C(=C1)C(=O)C=CN2,6/000000083,InChI=1S/C12H12N2O/c13-6-1-2-9-3-4-11-10(8-9)1...,C12H12N2O,200,1.51,3.01,-9.66,3.58,-80,NC/C=C/c1ccc2[nH]ccc(=O)c2c1
296,OC(=O)C/C=C/C1C=CC=C(C=1Cl)O,4/000000101,InChI=1S/C10H9ClO3/c11-10-7(3-1-5-8(10)12)4-2-...,C10H9O3Cl,212,2.27,3.01,-2.70,2.22,0,O=C(O)C/C=C/c1cccc(O)c1Cl
297,OC(=O)/C=C/CCC1=NC2=C(N1)N=CC=C2,4/000000122,InChI=1S/C11H11N3O2/c15-10(16)6-2-1-5-9-13-8-4...,C11H11N3O2,217,1.00,3.01,-10.12,4.81,-100,O=C(O)/C=C/CCc1nc2cccnc2[nH]1
298,OC1=CC=CC(=C1N1C=C[N]C1=[NH2])I,2/000000087,InChI=1S/C9H8IN3O/c10-6-2-1-3-7(14)8(6)13-5-4-...,C9H8N3OI,301,1.93,3.01,-6.51,13.40,-120,Error 1


In [ ]:
#df["Smiles_curated"] = [curation(i,j) for i,j in zip(df["Smiles"], df.index)]


In [ ]:
# Delate smiles that rdkit could not read
print(df.shape)
df = df[df["SMILES_curated"] != "Error 1"]
print(df.shape)
# Delate smiles that no contain allowed atoms
df = df[df["SMILES_curated"] != "Error 2"]
print(df.shape)
# Delate other errors
df = df[df["SMILES_curated"] != "Something else was found"].reset_index(drop=True)
print(df.shape)

(300, 11)
(291, 11)
(291, 11)
(291, 11)


In [ ]:
#Eliminate the empty rows from the column 'Smiles_curated'
#Since the generated SMILES strings from the curation process are canonical won't be two different molecules with the same SMILES string.
df= df.dropna(subset=['SMILES_curated'])

# Delete duplicates
df = df.drop_duplicates(subset=["SMILES_curated"], keep="first").reset_index(drop=True)
print(df.shape)
df.head(10)

(291, 11)


,SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical,SMILES_curated
0,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NCC1=CC=CC3=C1C...,3/000000487,InChI=1S/C21H17N3O5/c25-16(22-10-12-4-2-5-14-1...,C21H17N3O5,391,1.84,5.39,-8.34,6.45,-60,O=C(C/C=C1\Cc2cccc(C(=O)O)c2N1)NCc1cccc2c1C(=O...
1,ClC1=CC=C(C=C1)COCCNC(=O)C/C=C\1/NC2C(C1)=CC=C...,3/000000194,InChI=1S/C22H25ClN2O2/c1-2-16-3-6-18-14-20(25-...,C22H25N2O2Cl,384,4.17,4.95,-7.42,6.45,-100,CCc1ccc2c(c1)N/C(=C/CC(=O)NCCOCc1ccc(Cl)cc1)C2
2,C1C(OCC2=NC=CC(=C2)C(=O)N)=CC(=CC=1)N1C(=CCNC1...,3/000000246,InChI=1S/C18H18N4O4/c19-17(23)12-4-7-21-13(8-1...,C18H17N4O4,353,0.87,4.92,-2.59,22.40,-180,NC(=O)c1ccnc(COc2cccc(N3CNCC=C3C(=O)O)c2)c1
3,C1=NC2C(C(=O)O1)=CC=CC=2NC(=O)C/C=C\1/NC2C(C1)...,3/000000066,InChI=1S/C21H19N3O3/c1-2-13-6-7-14-11-15(23-18...,C21H19N3O3,361,2.80,4.90,-6.57,6.45,-100,CCc1ccc2c(c1)N/C(=C/CC(=O)Nc1cccc3c(=O)ocnc13)C2
4,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NC[C@@H](CO)F)/C2,3/000000466,InChI=1S/C15H17FN2O4/c16-10(8-19)7-17-13(20)5-...,C15H17N2O4F,308,0.96,4.90,-2.45,4.94,-190,O=C(C/C=C1\Cc2cccc(C(=O)O)c2N1)NC[C@H](F)CO
5,N(C(=O)C/C=C\1/NC2C(C1)=CC=C(C=2)CC)C1C=CC=C(C...,3/000000113,InChI=1S/C21H22ClN3O2/c1-2-14-6-7-15-11-18(23-...,C21H22N3O2Cl,383,3.08,4.88,-7.46,6.45,-60,CCc1ccc2c(c1)N/C(=C/CC(=O)Nc1cccc(NC(=O)CCl)c1)C2
6,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NCC1=CN=CN=C1)/C2,3/000000450,InChI=1S/C17H16N4O3/c22-15(20-9-11-7-18-10-19-...,C17H16N4O3,324,0.58,4.78,-7.58,5.09,-40,O=C(C/C=C1\Cc2cccc(C(=O)O)c2N1)NCc1cncnc1
7,OC1N=CC=C(C=1)COC1C=CC(=C(C=1)N1C(=CCNC1)C(=O)...,3/000000376,InChI=1S/C18H18BrN3O4/c19-9-13-1-2-14(26-10-12...,C18H17N3O4Br,419,3.01,4.71,-4.07,21.70,-180,O=C(O)C1=CCNCN1c1cc(OCc2ccnc(O)c2)ccc1CBr
8,C1=CN=C(N=C1CCNC(=O)C/C=C\1/NC2C(C1)=CC=C(C=2)...,3/000000201,InChI=1S/C19H23N5O/c1-2-13-3-4-14-12-16(23-17(...,C19H23N5O,337,1.60,4.71,-3.31,5.09,-60,CCc1ccc2c(c1)N/C(=C/CC(=O)NCCc1ccnc(N)n1)C2
9,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NCCCCC(=O)O)/C2,3/000000481,InChI=1S/C17H20N2O5/c20-14(18-9-2-1-6-15(21)22...,C17H20N2O5,332,1.42,4.70,-7.13,5.09,-40,O=C(O)CCCCNC(=O)C/C=C1\Cc2cccc(C(=O)O)c2N1


In [ ]:
#Add ID column
df.insert(0, 'ID_deNovo', [f"HD8_DL{str(i+1).zfill(3)}" for i in range(len(df))])
df

,ID_deNovo,SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical,SMILES_curated
0,HD8_DL001,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NCC1=CC=CC3=C1C...,3/000000487,InChI=1S/C21H17N3O5/c25-16(22-10-12-4-2-5-14-1...,C21H17N3O5,391,1.84,5.39,-8.34,6.45,-60,O=C(C/C=C1\Cc2cccc(C(=O)O)c2N1)NCc1cccc2c1C(=O...
1,HD8_DL002,ClC1=CC=C(C=C1)COCCNC(=O)C/C=C\1/NC2C(C1)=CC=C...,3/000000194,InChI=1S/C22H25ClN2O2/c1-2-16-3-6-18-14-20(25-...,C22H25N2O2Cl,384,4.17,4.95,-7.42,6.45,-100,CCc1ccc2c(c1)N/C(=C/CC(=O)NCCOCc1ccc(Cl)cc1)C2
2,HD8_DL003,C1C(OCC2=NC=CC(=C2)C(=O)N)=CC(=CC=1)N1C(=CCNC1...,3/000000246,InChI=1S/C18H18N4O4/c19-17(23)12-4-7-21-13(8-1...,C18H17N4O4,353,0.87,4.92,-2.59,22.40,-180,NC(=O)c1ccnc(COc2cccc(N3CNCC=C3C(=O)O)c2)c1
3,HD8_DL004,C1=NC2C(C(=O)O1)=CC=CC=2NC(=O)C/C=C\1/NC2C(C1)...,3/000000066,InChI=1S/C21H19N3O3/c1-2-13-6-7-14-11-15(23-18...,C21H19N3O3,361,2.80,4.90,-6.57,6.45,-100,CCc1ccc2c(c1)N/C(=C/CC(=O)Nc1cccc3c(=O)ocnc13)C2
4,HD8_DL005,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NC[C@@H](CO)F)/C2,3/000000466,InChI=1S/C15H17FN2O4/c16-10(8-19)7-17-13(20)5-...,C15H17N2O4F,308,0.96,4.90,-2.45,4.94,-190,O=C(C/C=C1\Cc2cccc(C(=O)O)c2N1)NC[C@H](F)CO
...,...,...,...,...,...,...,...,...,...,...,...,...
286,HD8_DL287,OC(=O)C1=NN=C(O1)CC#CC(O)(F)F,6/000000013,"InChI=1S/C7H4F2N2O4/c8-7(9,14)3-1-2-4-10-11-5(...",C7H4N2O4F2,218,0.57,3.02,-5.31,24.20,-190,O=C(O)c1nnc(CC#CC(O)(F)F)o1
287,HD8_DL288,C(=C\CN)/C1=CC=C2C(=C1)C(=O)C=CN2,6/000000083,InChI=1S/C12H12N2O/c13-6-1-2-9-3-4-11-10(8-9)1...,C12H12N2O,200,1.51,3.01,-9.66,3.58,-80,NC/C=C/c1ccc2[nH]ccc(=O)c2c1
288,HD8_DL289,OC(=O)C/C=C/C1C=CC=C(C=1Cl)O,4/000000101,InChI=1S/C10H9ClO3/c11-10-7(3-1-5-8(10)12)4-2-...,C10H9O3Cl,212,2.27,3.01,-2.70,2.22,0,O=C(O)C/C=C/c1cccc(O)c1Cl
289,HD8_DL290,OC(=O)/C=C/CCC1=NC2=C(N1)N=CC=C2,4/000000122,InChI=1S/C11H11N3O2/c15-10(16)6-2-1-5-9-13-8-4...,C11H11N3O2,217,1.00,3.01,-10.12,4.81,-100,O=C(O)/C=C/CCc1nc2cccnc2[nH]1


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
df.to_csv("/content/drive/MyDrive/FBDD_Denovo_VRC/smHDAC8_Prueba_all_1000_Druglike/Analisis de resultados/DB_1000_Druglike/DB_Druglike_curada.csv", sep=";", index=False)



Mounted at /content/drive
